In [1]:
import pandas as pd
import numpy as np
import joblib
import tensorflow as tf
import shap

logreg = joblib.load('../models/logreg.pkl')
rf = joblib.load('../models/random_forest.pkl')
nn = tf.keras.models.load_model('../models/neural_network.keras')

X_test_scaled = pd.read_csv('../data/X_test_scaled.csv')
X_test_unscaled = pd.read_csv('../data/X_test_unscaled.csv')
y_test = pd.read_csv('../data/y_test.csv').squeeze()

In [2]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

y_proba_lr = logreg.predict_proba(X_test_scaled)[:, 1]
y_proba_rf = rf.predict_proba(X_test_unscaled)[:, 1]
y_proba_nn = nn.predict(X_test_scaled).flatten()

def metrics_row(name, y_proba, threshold=0.5):
    y_pred = (y_proba >= threshold).astype(int)
    return {
        'Model': name,
        'Accuracy': round(accuracy_score(y_test, y_pred), 3),
        'Precision': round(precision_score(y_test, y_pred), 3),
        'Recall': round(recall_score(y_test, y_pred), 3),
        'F1': round(f1_score(y_test, y_pred), 3),
        'ROC-AUC': round(roc_auc_score(y_test, y_proba), 3),
    }

comparison = pd.DataFrame([
    metrics_row('Logistic Regression', y_proba_lr),
    metrics_row('Random Forest', y_proba_rf),
    metrics_row('Neural Network', y_proba_nn),
])
comparison

10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 


,Model,Accuracy,Precision,Recall,F1,ROC-AUC
0,Logistic Regression,0.857,0.571,0.426,0.488,0.721
1,Random Forest,0.806,0.375,0.319,0.345,0.731
2,Neural Network,0.857,0.571,0.426,0.488,0.739


In [3]:
comparison['Explainability Method'] = ['Native coefficients', 'SHAP TreeExplainer', 'SHAP KernelExplainer']
comparison['Explainability Quality (1-5)'] = [5, 4, 2]
comparison['Regulatory Suitability'] = ['High - fully transparent', 'Medium - post-hoc, exact', 'Low - post-hoc, approximate']
comparison['Computational Cost'] = ['Very low', 'Low-Medium', 'High']
comparison['Maintenance Complexity'] = ['Low', 'Medium', 'High']
comparison.to_csv('../data/model_comparison.csv', index=False)
comparison

,Model,Accuracy,Precision,Recall,F1,ROC-AUC,Explainability Method,Explainability Quality (1-5),Regulatory Suitability,Computational Cost,Maintenance Complexity
0,Logistic Regression,0.857,0.571,0.426,0.488,0.721,Native coefficients,5,High - fully transparent,Very low,Low
1,Random Forest,0.806,0.375,0.319,0.345,0.731,SHAP TreeExplainer,4,"Medium - post-hoc, exact",Low-Medium,Medium
2,Neural Network,0.857,0.571,0.426,0.488,0.739,SHAP KernelExplainer,2,"Low - post-hoc, approximate",High,High


In [6]:
print(shap_values_rf.shape)

(20, 44, 2)


In [5]:
import numpy as np

def faithfulness_test(model_predict_fn, X, shap_values, n_samples=20):
    """Checks whether zeroing-out the top SHAP feature moves the prediction in the expected direction."""
    # If shap_values is 3D (samples, features, classes), keep only the positive class
    if shap_values.ndim == 3:
        shap_values = shap_values[:, :, 1]

    results = []
    for i in range(n_samples):
        row = X.iloc[[i]].copy()
        original_pred = model_predict_fn(row)[0]

        top_idx = np.argmax(np.abs(shap_values[i]))
        top_feature_col = X.columns[top_idx]
        shap_sign = np.sign(shap_values[i][top_idx])

        perturbed = row.copy()
        perturbed[top_feature_col] = X[top_feature_col].mean()
        new_pred = model_predict_fn(perturbed)[0]

        moved_as_expected = np.sign(original_pred - new_pred) == shap_sign
        results.append(moved_as_expected)
    return np.mean(results)

# Example usage for Random Forest
explainer_rf = shap.TreeExplainer(rf)
shap_values_rf = explainer_rf.shap_values(X_test_unscaled.iloc[:20])

rf_predict_fn = lambda X: rf.predict_proba(X)[:, 1]
faithfulness_score = faithfulness_test(rf_predict_fn, X_test_unscaled.iloc[:20], shap_values_rf)
print(f"Random Forest SHAP faithfulness score: {faithfulness_score:.2%}")

Random Forest SHAP faithfulness score: 55.00%


In [7]:
def stability_test(explainer, X, n_pairs=10, noise_scale=0.01):
    diffs = []
    for i in range(n_pairs):
        row = X.iloc[[i]]
        noisy_row = row + np.random.normal(0, noise_scale, row.shape)
        sv1 = explainer.shap_values(row)
        sv2 = explainer.shap_values(noisy_row)
        diffs.append(np.mean(np.abs(np.array(sv1) - np.array(sv2))))
    return np.mean(diffs)

stability_score_rf = stability_test(explainer_rf, X_test_unscaled.iloc[:10])
print(f"Random Forest SHAP stability (lower = more stable): {stability_score_rf:.4f}")

Random Forest SHAP stability (lower = more stable): 0.0000


In [8]:
import numpy as np

print(shap_values_rf.shape)  # run this first to confirm — likely (n_samples, 44, 2)

(20, 44, 2)


In [9]:
# Handle both possible SHAP output formats
if shap_values_rf.ndim == 3:
    # shape is (n_samples, n_features, n_classes) -> take class 1 (attrition = Yes)
    shap_vals_class1 = shap_values_rf[:, :, 1]
else:
    # older shap versions: shap_values_rf is already 2D for the positive class
    shap_vals_class1 = shap_values_rf

top_features_lr = pd.Series(np.abs(logreg.coef_[0]), index=X_test_scaled.columns).nlargest(10).index.tolist()
top_features_rf = pd.Series(np.abs(shap_vals_class1).mean(axis=0), index=X_test_unscaled.columns).nlargest(10).index.tolist()

overlap = set(top_features_lr) & set(top_features_rf)
print(f"Top-10 feature overlap between LogReg and RF: {len(overlap)}/10")
print(overlap)

Top-10 feature overlap between LogReg and RF: 3/10
{'StockOptionLevel', 'Department_Research & Development', 'EducationField_Medical'}


In [11]:
explainer_nn = shap.KernelExplainer(nn.predict, shap.sample(X_test_scaled, 50))
nn_predict_fn = lambda X: nn.predict(X, verbose=0).flatten()
shap_values_nn_small = np.array(explainer_nn.shap_values(X_test_scaled.iloc[:20]))

# NN's shap values come back as (n_samples, n_features, 1) - single class, squeeze it
if shap_values_nn_small.ndim == 3:
    shap_values_nn_small = shap_values_nn_small[:, :, 0]

faithfulness_score_nn = faithfulness_test(nn_predict_fn, X_test_scaled.iloc[:20], shap_values_nn_small)
print(f"Neural Network SHAP faithfulness score: {faithfulness_score_nn:.2%}")

stability_score_nn = stability_test(explainer_nn, X_test_scaled.iloc[:10])
print(f"Neural Network SHAP stability (lower = more stable): {stability_score_nn:.4f}")

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step


  0%|          | 0/20 [00:00<?, ?it/s]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
3335/3335 ━━━━━━━━━━━━━━━━━━━━ 2s 488us/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
3335/3335 ━━━━━━━━━━━━━━━━━━━━ 2s 532us/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
3335/3335 ━━━━━━━━━━━━━━━━━━━━ 2s 533us/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
3335/3335 ━━━━━━━━━━━━━━━━━━━━ 2s 515us/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
3335/3335 ━━━━━━━━━━━━━━━━━━━━ 2s 550us/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
3335/3335 ━━━━━━━━━━━━━━━━━━━━ 2s 725us/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
3335/3335 ━━━━━━━━━━━━━━━━━━━━ 3s 886us/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
3335/3335 ━━━━━━━━━━━━━━━━━━━━ 2s 741us/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
3335/3335 ━━━━━━━━━━━━━━━━━━━━ 3s 918us/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
3335/3335 ━━━━━━━━━━━━━━━━━━━━ 3s 930us/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
3335/3335 ━━━━━━━━━━━━━━━━━━━━ 3s 781us/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
3335/3335 ━━━━━━━━━━━━━━━━━━━━ 3s 796us/step
1/1 

  0%|          | 0/1 [00:00<?, ?it/s]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
3335/3335 ━━━━━━━━━━━━━━━━━━━━ 3s 955us/step


...

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
3338/3338 ━━━━━━━━━━━━━━━━━━━━ 2s 721us/step


  0%|          | 0/1 [00:00<?, ?it/s]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
3335/3335 ━━━━━━━━━━━━━━━━━━━━ 3s 779us/step


  0%|          | 0/1 [00:00<?, ?it/s]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
3338/3338 ━━━━━━━━━━━━━━━━━━━━ 2s 717us/step
Neural Network SHAP stability (lower = more stable): 0.0023


## Model Selection Recommendation

**Context:** Three models were evaluated for deployment in an HR attrition risk-scoring use case, audited against accuracy, explainability quality, and regulatory suitability per RBI FREE-AI's "Understandable by Design" Sutra and EU AI Act high-risk system transparency requirements (employment decisions are explicitly high-risk under the Act).

**Recommendation:** Logistic Regression and Neural Network are statistically tied on F1/precision/recall (0.488), with the Neural Network showing a modest ROC-AUC edge (0.739 vs 0.721). Random Forest underperforms both on every threshold-based metric despite a comparable AUC, likely due to conservative hyperparameters (max_depth=8, min_samples_leaf=5) limiting its ability to exploit non-linear patterns. Given Logistic Regression matches the Neural Network's practical performance while offering full native interpretability, it is the recommended deployment choice — the Neural Network's marginal AUC gain does not justify its explainability cost.

**Explainability cost evidence**: SHAP faithfulness testing showed Logistic Regression's explanation is the model (100% faithful by construction), Random Forest's TreeExplainer achieved 55% faithfulness, while the Neural Network's KernelExplainer achieved 100% faithfulness on this sample — though on a much smaller perturbation sample (n=20) and using a 50-row background, so this single high score should be treated cautiously rather than as proof KernelExplainer is reliably faithful. Stability testing showed Random Forest's SHAP values are highly stable under small input perturbations (mean diff ≈0.0000) versus the Neural Network's KernelExplainer (≈0.0023) — over an order of magnitude less stable, consistent with the conceptual claim that sampling-based approximation methods produce less reproducible explanations than exact tree-based methods.

**Deployment conditions:** Human review required for any individual flagged above the high-risk threshold before action is taken (aligned with RBI's human-override principle). Quarterly bias audits across gender, age, and marital status fields. SHAP explanations must accompany every individual prediction surfaced to an HR manager.

**Monitoring requirements:** Track prediction drift and SHAP feature-importance drift quarterly; retrain if attrition base rate shifts by more than 5 percentage points.

**Caveats:** This dataset is IBM's synthetic/illustrative HR dataset, not live production data — bias and fairness findings here are illustrative of methodology, not a real audit of an actual organization's HR practices.